# TP 3 — Entraîner un mini-GPT avec nanoGPT — CORRIGÉ

## Objectif général

Ce corrigé montre comment passer du code modèle à un vrai entraînement, même très court.  
Le but n'est pas d'obtenir un bon Shakespeare, mais de comprendre le cycle :

données → batches → forward → loss → backward → optimizer → checkpoint.


## Sources principales du parcours

Ces notebooks se basent principalement sur nanoGPT de Karpathy et sur la documentation officielle PyTorch/Jupyter.

- nanoGPT — README et quick start : https://github.com/karpathy/nanoGPT#quick-start  
  À lire pour comprendre le flux officiel : préparer Shakespeare, entraîner avec `train.py`, puis générer avec `sample.py`.
- nanoGPT — `model.py` : https://github.com/karpathy/nanoGPT/blob/master/model.py  
  À lire comme colonne vertébrale : `LayerNorm`, `CausalSelfAttention`, `MLP`, `Block`, `GPTConfig`, `GPT.forward`, `GPT.generate`.
- nanoGPT — `train.py` : https://github.com/karpathy/nanoGPT/blob/master/train.py  
  À lire pour la boucle d'entraînement : batch, forward, loss, backward, optimizer, checkpoint.
- nanoGPT — `sample.py` : https://github.com/karpathy/nanoGPT/blob/master/sample.py  
  À lire pour l'inférence : chargement du checkpoint, encodage du prompt, `model.generate`.
- Config Shakespeare caractère : https://github.com/karpathy/nanoGPT/blob/master/config/train_shakespeare_char.py  
  À lire pour comprendre les hyperparamètres pédagogiques : `block_size`, `n_layer`, `n_head`, `n_embd`, `dropout`, `max_iters`.
- PyTorch — Tensors : https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html  
  À lire pour comprendre pourquoi les données, poids et activations sont des tenseurs.
- PyTorch — `torch.nn.Embedding` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html  
  À lire pour comprendre la table qui transforme des indices de tokens en vecteurs.
- PyTorch — `torch.nn.LayerNorm` : https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html  
  À lire pour comprendre la normalisation utilisée dans chaque bloc.
- PyTorch — `torch.nn.GELU` : https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html  
  À lire pour comprendre l'activation du MLP.
- PyTorch — `torch.nn.CrossEntropyLoss` : https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html  
  À lire pour comprendre la loss de prédiction du prochain token.
- PyTorch — `torch.nn.functional.scaled_dot_product_attention` : https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html  
  À lire pour comprendre la version optimisée de l'attention Q/K/V.
- PyTorch — CUDA semantics : https://docs.pytorch.org/docs/stable/notes/cuda.html  
  À lire pour comprendre le placement CPU/GPU des tenseurs et modèles.
- Jupyter — Markdown cells : https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html  
  À lire pour savoir structurer les réponses dans les cellules Markdown.


## Partie 1 — Préparer les données

### Question 3.1 — Préparer Shakespeare

**Solution :**


In [ ]:
from pathlib import Path
import subprocess, sys, os, torch

repo = Path("nanoGPT")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/karpathy/nanoGPT.git"], check=True)

subprocess.run([sys.executable, "data/shakespeare_char/prepare.py"], cwd=repo, check=True)

for name in ["train.bin", "val.bin", "meta.pkl"]:
    print(name, "->", (repo / "data/shakespeare_char" / name).exists())


### Explication pédagogique

Le texte brut est transformé en une longue suite d'entiers.  
`train.py` ne lit pas des phrases : il lit des fragments de cette suite d'entiers.

Cette préparation permet d'échantillonner rapidement des batches pendant l'entraînement.


### Question 3.2 — Lire la config Shakespeare

**Solution :**


In [ ]:
from pathlib import Path

config_path = Path("nanoGPT/config/train_shakespeare_char.py")
text = config_path.read_text(encoding="utf-8")
print(text)


**Réponse attendue :**

Dans la config officielle :
- `dataset = 'shakespeare_char'`
- `block_size = 256`
- `n_layer = 6`
- `n_head = 6`
- `n_embd = 384`

### Explication pédagogique

Ces valeurs définissent un “baby GPT”.  
Il est assez petit pour être pédagogique, mais assez réel pour contenir les mêmes composants qu'un GPT plus grand : embeddings, attention causale, MLP, empilement de blocs.

Dans ce TP, on va encore réduire ces valeurs pour accélérer l'exécution CPU.


## Partie 2 — Lancer un entraînement court

### Question 3.3 — Lancer un entraînement pédagogique court

**Solution :**


In [ ]:
from pathlib import Path
import subprocess, sys, os

repo = Path("nanoGPT")
cmd = [
    sys.executable, "train.py", "config/train_shakespeare_char.py",
    "--device=cpu",
    "--compile=False",
    "--max_iters=50",
    "--eval_interval=25",
    "--eval_iters=5",
    "--log_interval=10",
    "--batch_size=8",
    "--block_size=32",
    "--n_layer=2",
    "--n_head=2",
    "--n_embd=64",
    "--dropout=0.0",
    "--out_dir=out-shakespeare-char-tp",
    "--always_save_checkpoint=True",
]

print("Commande :")
print(" ".join(cmd))
subprocess.run(cmd, cwd=repo, check=True)


### Explication pédagogique

On garde la structure GPT mais on réduit :
- le contexte (`block_size=32`) ;
- le batch (`batch_size=8`) ;
- le nombre de couches (`n_layer=2`) ;
- le nombre de têtes (`n_head=2`) ;
- la dimension interne (`n_embd=64`) ;
- le nombre d'itérations (`max_iters=50`).

Ce modèle ne générera pas un bon texte. C'est normal.  
Il sert à observer la mécanique complète sans attendre longtemps.

**Point AI Engineer :**  
Quand tu diminues `block_size`, tu réduis fortement le coût de l'attention, car la matrice d'attention dépend de `T²`.


### Question 3.4 — Comprendre les logs d'entraînement

**Solution :**

- `iter` : numéro de l'itération d'entraînement.
- `loss` : perte calculée sur le batch courant.
- `train loss` : perte moyenne estimée sur plusieurs batches du split train.
- `val loss` : perte moyenne estimée sur plusieurs batches du split validation.
- `time` : temps pris par une itération, souvent en millisecondes.

### Explication pédagogique

La loss mesure à quel point le modèle se trompe dans la prédiction du prochain token.  
Une loss qui baisse signifie que le modèle apprend à mieux prédire les caractères suivants.

La différence entre train et validation est importante :
- train loss : performance sur les données utilisées pour apprendre ;
- val loss : performance sur des données gardées pour évaluer la généralisation.

**Erreur fréquente :**  
Ne regarde pas seulement la loss d'un batch. Elle peut être bruitée. La validation donne une vision plus stable.


## Partie 3 — Vérifier le checkpoint

### Question 3.5 — Vérifier `ckpt.pt`

**Solution :**


In [ ]:
from pathlib import Path
import torch

ckpt_path = Path("nanoGPT/out-shakespeare-char-tp/ckpt.pt")
print("Checkpoint existe :", ckpt_path.exists())
assert ckpt_path.exists(), "Le checkpoint n'existe pas. Relance l'entraînement."

checkpoint = torch.load(ckpt_path, map_location="cpu")
print("Clés du checkpoint :")
for k in checkpoint.keys():
    print("-", k)

print("\nmodel_args :")
print(checkpoint["model_args"])
print("\niter_num :", checkpoint["iter_num"])
print("best_val_loss :", checkpoint["best_val_loss"])


### Explication pédagogique

Un checkpoint n'est pas seulement un fichier de poids.

Il contient généralement :
- les poids du modèle (`model`) ;
- l'état de l'optimiseur (`optimizer`) ;
- les arguments nécessaires pour reconstruire le modèle (`model_args`) ;
- l'itération actuelle ;
- la meilleure loss de validation ;
- la configuration utilisée.

**À quoi ça sert ?**  
Cela permet de reprendre l'entraînement ou de faire de l'inférence plus tard.


### Question 3.6 — Reprendre depuis un checkpoint

**Solution :**


In [ ]:
from pathlib import Path
import subprocess, sys

repo = Path("nanoGPT")
cmd = [
    sys.executable, "train.py", "config/train_shakespeare_char.py",
    "--device=cpu",
    "--compile=False",
    "--init_from=resume",
    "--out_dir=out-shakespeare-char-tp",
    "--max_iters=60",
    "--eval_interval=10",
    "--eval_iters=5",
    "--log_interval=5",
]
print("Commande :")
print(" ".join(cmd))
subprocess.run(cmd, cwd=repo, check=True)


### Explication pédagogique

`--init_from=resume` demande à `train.py` de charger `ckpt.pt` depuis `out_dir`.

Attention : pour reprendre correctement, les paramètres structurants doivent correspondre :
- `n_layer`
- `n_head`
- `n_embd`
- `block_size`
- `vocab_size`
- `bias`

Sinon, les poids sauvegardés ne correspondent plus à l'architecture reconstruite.

**Erreur fréquente :**  
Changer la taille du modèle au moment de reprendre. Tu ne peux pas charger des poids d'un modèle `(n_embd=64)` dans un modèle `(n_embd=128)` sans opération spéciale.
